# Encoding Technique 2: One-Hot Encoding

**Dataset:** `Loan_Default.csv`

**When to use:** For **nominal** (unordered) categorical features with a **manageable number of unique categories**.

**Key concept:** For a feature with N unique categories, OHE creates **N new binary columns** — one per category. A `1` means that category is present; `0` means it's absent.

**Trade-off:** Increases dimensionality significantly. Can cause multicollinearity (see Dummy Encoding as the fix).

This notebook demonstrates two approaches:
- `category_encoders.OneHotEncoder` (more flexible)
- `sklearn.preprocessing.OneHotEncoder` (standard scikit-learn approach)

---

### Step 1: Setup, Data Loading & Prep

In [1]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
import category_encoders as ce

# Load data from the raw directory
df = pd.read_csv('../../data/raw/Loan_Default.csv')
df.drop(['ID', 'year'], axis=1, inplace=True)

categorical_features = df.select_dtypes(include=['object']).columns.tolist()
Ordinal_features = ['age']
Nominal_features = categorical_features.copy()
Nominal_features.remove('age')

# Pre-encode the ordinal feature so the baseline data is clean
enc = OrdinalEncoder()
df[Ordinal_features] = enc.fit_transform(df[Ordinal_features])

print(f'Shape before OHE: {df.shape}')
df.head()

Shape before OHE: (148670, 32)


,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,loan_amount,rate_of_interest,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,116500,NaN,...,EXP,758,CIB,0.0,to_inst,98.728814,south,direct,1,45.0
1,cf,Male,nopre,type2,p1,l1,nopc,b/c,206500,NaN,...,EQUI,552,EXP,3.0,to_inst,NaN,North,direct,1,NaN
2,cf,Male,pre,type1,p1,l1,nopc,nob/c,406500,4.56,...,EXP,834,CIB,1.0,to_inst,80.019685,south,direct,0,46.0
3,cf,Male,nopre,type1,p4,l1,nopc,nob/c,456500,4.25,...,EXP,587,CIB,2.0,not_inst,69.376900,North,direct,0,42.0
4,cf,Joint,pre,type1,p1,l1,nopc,nob/c,696500,4.00,...,CRIF,602,EXP,0.0,not_inst,91.886544,North,direct,0,39.0


---
## Approach A: `category_encoders.OneHotEncoder`

In [2]:
df_onehot_ce = df.copy()

# Configure the encoder
OH_encoder = ce.OneHotEncoder(
    cols=Nominal_features,
    handle_unknown='return_nan',
    return_df=True,
    use_cat_names=True   # gives descriptive names like 'Gender_Male'
)

In [3]:
# Separate and encode
df_onehot_ce_numerical = df_onehot_ce.drop(Nominal_features, axis=1)
df_onehot_ce_categorical = OH_encoder.fit_transform(df_onehot_ce[Nominal_features])

# Reassemble
df_onehot_ce = pd.concat([df_onehot_ce_numerical, df_onehot_ce_categorical], axis=1)

print(f'Shape after OHE (category_encoders): {df_onehot_ce.shape}')
df_onehot_ce.columns.tolist()

Shape after OHE (category_encoders): (148670, 69)


['loan_amount',
 'rate_of_interest',
 'Interest_rate_spread',
 'Upfront_charges',
 'term',
 'property_value',
 'income',
 'Credit_Score',
 'age',
 'LTV',
 'Status',
 'dtir1',
 'loan_limit_cf',
 'loan_limit_ncf',
 'loan_limit_nan',
 'Gender_Sex Not Available',
 'Gender_Male',
 'Gender_Joint',
 'Gender_Female',
 'approv_in_adv_nopre',
 'approv_in_adv_pre',
 'approv_in_adv_nan',
 'loan_type_type1',
 'loan_type_type2',
 'loan_type_type3',
 'loan_purpose_p1',
 'loan_purpose_p4',
 'loan_purpose_p3',
 'loan_purpose_p2',
 'loan_purpose_nan',
 'Credit_Worthiness_l1',
 'Credit_Worthiness_l2',
 'open_credit_nopc',
 'open_credit_opc',
 'business_or_commercial_nob/c',
 'business_or_commercial_b/c',
 'Neg_ammortization_not_neg',
 'Neg_ammortization_neg_amm',
 'Neg_ammortization_nan',
 'interest_only_not_int',
 'interest_only_int_only',
 'lump_sum_payment_not_lpsm',
 'lump_sum_payment_lpsm',
 'construction_type_sb',
 'construction_type_mh',
 'occupancy_type_pr',
 'occupancy_type_sr',
 'occupancy_type

---
## Approach B: `sklearn.preprocessing.OneHotEncoder`

In [4]:
df_onehot_sk = df.copy()

OH_encoder_sk = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
df_onehot_sk_categorical = pd.DataFrame(OH_encoder_sk.fit_transform(df_onehot_sk[Nominal_features]))

# Assign descriptive column names
df_onehot_sk_categorical.columns = OH_encoder_sk.get_feature_names_out(Nominal_features)
df_onehot_sk_categorical.index = df_onehot_sk.index

# Remove original categorical columns and add encoded ones
df_onehot_sk_numerical = df_onehot_sk.drop(Nominal_features, axis=1)
df_onehot_sk = pd.concat([df_onehot_sk_numerical, df_onehot_sk_categorical], axis=1)

print(f'Shape after OHE (sklearn): {df_onehot_sk.shape}')
df_onehot_sk.head()

Shape after OHE (sklearn): (148670, 69)


,loan_amount,rate_of_interest,Interest_rate_spread,Upfront_charges,term,property_value,income,Credit_Score,age,LTV,...,co-applicant_credit_type_EXP,submission_of_application_not_inst,submission_of_application_to_inst,submission_of_application_nan,Region_North,Region_North-East,Region_central,Region_south,Security_Type_Indriect,Security_Type_direct
0,116500,NaN,NaN,NaN,360.0,118000.0,1740.0,758,0.0,98.728814,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
1,206500,NaN,NaN,NaN,360.0,NaN,4980.0,552,3.0,NaN,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,406500,4.56,0.2000,595.0,360.0,508000.0,9480.0,834,1.0,80.019685,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
3,456500,4.25,0.6810,NaN,360.0,658000.0,11880.0,587,2.0,69.376900,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,696500,4.00,0.3042,0.0,360.0,758000.0,10440.0,602,0.0,91.886544,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


### Key Observation

Notice how the number of columns **increased dramatically** after One-Hot Encoding. This is the "curse of dimensionality" trade-off. For the `Gender` column alone (4 categories), OHE created 4 new binary columns.